# Your first IETS spectrum

Tunneling electrons can excite a spin when their available energy exceeds a level splitting. Opening this transport channel produces steps in differential conductance, $dI/dV$.

We calculate a spin-half at 5 T, first with thermal populations and then with tunneling-induced pumping of the state populations.

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from spinfinity import SpinSys, dIdV, hamil, plotting

pio.renderers.default = "plotly_mimetype+notebook"
# Use "fixed" for compact plots, or "no" to suppress display.
plotting.set_display_mode("responsive")

## Hamiltonian and scattering parameters

For $g=2$ and a field along $z$, the splitting is approximately $0.579 \mathrm{meV}$. Excitation thresholds are therefore near $\pm0.579 \mathrm{mV}$.

`G_st` sets the transport conductance scale. `G_ss` controls substrate scattering without transferring charge between tip and sample. We disable tip-tip scattering with `G_tt=0`. `U=0` removes potential scattering; `b0=0` removes the separate spin-independent background.

In [2]:
spin = SpinSys([0.5])
spin.B = [0.0, 0.0, 5.0]  # T
spin.BTip = [0.0, 0.0, 0.0]  # T
spin.gfactorvector = [2.0]

# Tunneling parameters
spin.T = 0.5  # K
spin.G_st = 1e-9  # S
spin.G_ss = [0.1e-9]  # S
spin.G_tt = 0  # can be 0 to disable or 1 to enable
spin.TipPolarization = [0.0, 0.0, 0.0]
spin.U = np.array([0.0])
spin.b0 = 0.0

hamil.calc_EigEnergies(spin, rebuildH=True)
gap = spin.E_All[1] - spin.E_All[0]
voltage = np.linspace(-1.5, 1.5, 601)  # mV

print(f"Relative eigenenergies: {spin.E_All} meV")
print(f"Excitation thresholds: +/-{gap:.4f} mV")

Relative eigenenergies: [0.         0.57883818] meV
Excitation thresholds: +/-0.5788 mV


## Thermal second-order spectrum

`plot_IETS(order="2nd")` calculates a thermal spectrum and constructs its Plotly figure. Finite temperature broadens the steps.

With `norm=False`, conductance is returned in $\mathrm{S}$.

In [5]:
dIdV.plot_IETS(
    spin, V_array=voltage, order="2nd", norm=False,
);


## Tunneling-induced pumping

The order `2ndR` solves voltage-dependent population rate equations and numerically differentiates the current. Population redistribution changes the conductance and can produce an overshoot.

In [10]:
dIdV.plot_IETS(
    spin, V_array=voltage, order="2ndR", norm=False,
);